# Habitability predictor
## Trying to find if space dirt can grow plants

## 1: Imports

In [1]:
import csv
import pandas as pd

## 1a: Loading the data
### Data source:
Wamelink et al. (2014). Can Plants Grow on Mars and the Moon?
PLOS ONE. https://doi.org/10.1371/journal.pone.0103138 (open access)

table 2 from Wamelink 2014 was typed manually into soil_composition.csv file 

### What does each column mean:
| Column | What it is |
|---|---|
| pH | How acidic or alkaline the soil is  |
| N_NH4_mgkg | Ammonium — usable nitrogen, mg per kg of soil |
| N_NO3NO2_mgkg | Nitrate/nitrite — also usable nitrogen |
| P_PO4_mgkg | Usable phosphorus (almost always zero due to gap) |
| K_mgkg | Potassium — plant food |
| Al_mgkg | Aluminium — can be toxic in high amounts |
| C_total_gkg | Total carbon content |

In [2]:
df_soils = pd.read_csv("soil_composition.csv")
print("Soil composition data:")
df_soils

Soil composition data:


,soil,pH,N_NH4_mgkg,N_NO3NO2_mgkg,P_PO4_mgkg,K_mgkg,Al_mgkg,C_total_gkg
0,Earth,8.3,0.5,4.2,0.0,4.7,0.0,3.2
1,Moon,9.6,0.3,4.2,0.2,27.0,0.5,3.0
2,Mars,7.3,3.9,2.1,0.0,138.0,0.0,30.1


## 1b: Load the real plant growth experiment
This is **Supplementary Table S2** from Wamelink 2014.
840 rows — 14 plant species × 3 soils × 20 replica pots each.

**What each column means:**
| Column | What it is |
|---|---|
| Block | Which experimental block (1–20) |
| Number | Pot number |
| Species | Plant species name |
| Soil | Earth / Moon / Mars |
| nGerminated | How many seeds germinated (out of 5 per pot) |
| nLeaves | How many plants grew leaves |
| nFlowers | How many plants flowered |
| nAlive | How many plants survived 50 days |
| totalBiomass | Total dried plant weight in mg (the main outcome) |
| aboveGround | Above-ground biomass in mg |
| belowGround | Below-ground (root) biomass in mg |


In [3]:
df_exp = pd.read_excel("pone.0103138.s002.xlsx", sheet_name="Data")
print(f"Shape: {df_exp.shape} (rows, columns)")
print()
df_exp.head(10)

Shape: (840, 11) (rows, columns)



,Block,Number,Species,Soil,nGerminated,nLeaves,nFlowers,nAlive,totalBiomass,aboveGround,belowGround
0,1,2,M. officinalis,Mars,4,4,0,4,115.5,43.9,71.6
1,1,24,S. lycopersicum,Moon,2,0,0,0,5.3,5.3,0.1
2,1,36,V. sativa sativa,Moon,0,0,0,0,NaN,NaN,NaN
3,1,28,D. carota,Earth,4,3,0,3,32.7,13.4,19.3
4,1,15,S. reflexum,Moon,3,2,0,2,2.3,2.3,0.1
5,1,8,C. palustre,Mars,4,4,0,4,81.8,51.3,30.5
6,1,40,L. sativum,Earth,5,4,0,4,49.1,23.7,25.4
7,1,4,L. pendunculatus,Earth,5,4,0,5,44.5,14.7,29.8
8,1,9,C. palustre,Moon,2,2,0,0,4.8,1.9,2.9
9,1,22,S. lycopersicum,Earth,4,3,0,3,46.2,35.2,11.0


## 1c: Summarizing the outcomes
this is the **answer key**

In [4]:
summary = summary = df_exp.groupby("Soil").agg(
    mean_biomass_mg   = ("totalBiomass", "mean"),
    median_biomass_mg = ("totalBiomass", "median"),
    mean_germinated   = ("nGerminated",  "mean"),
    pots_with_data    = ("totalBiomass", "count"),
).round(2)

print("Real experiment results (answer key):")
print(summary)
print()
print("Real ranking from the research : mars > earth > moon")
print("This is what our rubric and model must reproduce.")

Real experiment results (answer key):
       mean_biomass_mg  median_biomass_mg  mean_germinated  pots_with_data
Soil                                                                      
Earth            61.57              36.75             2.99             244
Mars            131.42              76.75             3.30             258
Moon             35.77              10.60             2.51             223

Real ranking from the research : mars > earth > moon
This is what our rubric and model must reproduce.


## 2: Grading Rubric:

This rubric scores any soil's chemistry and return a number between 0 and 1 :

Higher means more plant friendly

### Features that are used :
pH: Controls whether ANY nutrients dissolve -  50% weighted 
Available N: Most differentiating signal across our 3 soils -  35% weighted
K: secondary but a real signal - 15 % weighted

### Features that are removed :
Phosporous (P): Near zero simulants across all 3 individual researchs (more in research_gaps file)

Sulphur (S): not measured in our database

### 2a: Functions for the rubric

In [6]:
"""
Takes a pH number, returns a score 0 to 1
Optimal score: 5.5 to 7.5
Linear ramp to 0 at pH 4.5 (too acidic) and pH 9.0 (too alkaline)
"""
def score_ph(ph):
    ph = float(ph)
    if ph <= 4.5 or ph >= 9.0:
        return 0.0
    if ph < 5.5:
        return (ph - 4.5) / (5.5 - 4.5)
    if ph <= 7.5:
        return 1.0
    return 1.0 - (ph - 7.5) / (9.0 - 7.5)

In [8]:
"""
Takes nitrogen numbers, returns 0 to 1
Available nitrogen = NH4 (ammonium) and NO3+NO2 (nitrate/nitrite).
Both forms are directly usable by plant roots.
Estimated reference max = 10 mg/kg  
"""
def score_n(nh4, no3):
    total_n = float(nh4) + float(no3)
    return min(1.0, total_n / 10.0)

In [10]:
"""
Takes potassium number, returns 0 to 1
Potassium. Estimated reference max = 150 mg/kg
"""
def score_k(k):
    return min(1.0, float(k) / 150.0)

#weights 
WEIGHTS = {
    "pH": 0.50,
    "N":  0.35,
    "K":  0.15,
}

In [11]:
"""
Calls all three and combines them into one final number.
Takes one row of soil data (as a dict or pandas Series).
Returns: (sub_scores dict, final_score float)
"""
def score_soil(row):
    subs = {
        "pH": score_ph(row["pH"]),
        "N":  score_n(row["N_NH4_mgkg"], row["N_NO3NO2_mgkg"]),
        "K":  score_k(row["K_mgkg"]),
    }
    total_w = sum(WEIGHTS.values())
    final   = sum(subs[k] * WEIGHTS[k] for k in subs) / total_w
    return subs, round(final, 3)

### 2b: running the rubric

In [12]:
records = []

for _, row in df_soils.iterrows():
    subs, final = score_soil(row)

    records.append({
        "Soil": row["soil"],
        "pH": round(subs["pH"], 2),
        "N": round(subs["N"], 2),
        "K": round(subs["K"], 2),
        "Final Score": final
    })

df_scores = pd.DataFrame(records).set_index("Soil")

print(df_scores)
print()

# Rank the soils
ranking = df_scores["Final Score"].sort_values(ascending=False)

print("Rubric ranking:", " > ".join(ranking.index))
print("Real ranking: Mars > Earth > Moon")

# Check if the rankings match
if ranking.index.tolist() == ["Mars", "Earth", "Moon"]:
    print("Match: YES")
else:
    print("Match: NO")

         pH     N     K  Final Score
Soil                                
Earth  0.47  0.47  0.03        0.403
Moon   0.00  0.45  0.18        0.184
Mars   1.00  0.60  0.92        0.848

Rubric ranking: Mars > Earth > Moon
Real ranking: Mars > Earth > Moon
Match: YES
